In [1]:
# =====================================================
# USDA NUTRITION DATASET JOINING
# =====================================================

import pandas as pd

# =====================================================
# FILE PATHS
# =====================================================

food_path = "/home/cnssec/Downloads/diet/datasets/FoodData_Central_foundation_food_csv_2026-04-30/food.csv"

food_nutrient_path = "/home/cnssec/Downloads/diet/datasets/FoodData_Central_foundation_food_csv_2026-04-30/food_nutrient.csv"

nutrient_path = "/home/cnssec/Downloads/diet/datasets/FoodData_Central_foundation_food_csv_2026-04-30/nutrient.csv"


In [3]:
print("Loading datasets...")

food_df = pd.read_csv(
    food_path,
    usecols=['fdc_id', 'description']
)

food_nutrient_df = pd.read_csv(
    food_nutrient_path,
    usecols=['fdc_id', 'nutrient_id', 'amount']
)

nutrient_df = pd.read_csv(
    nutrient_path,
    usecols=['id', 'name']
)


Loading datasets...


In [4]:
# =====================================================
# RENAME COLUMNS
# =====================================================

food_df.rename(columns={
    'description': 'food'
}, inplace=True)

nutrient_df.rename(columns={
    'id': 'nutrient_id',
    'name': 'nutrient_name'
}, inplace=True)



In [5]:
# =====================================================
# IMPORTANT NUTRIENTS TO KEEP
# =====================================================

important_nutrients = {
    'Energy': 'calories',
    'Protein': 'protein',
    'Carbohydrate, by difference': 'carbs',
    'Total lipid (fat)': 'fat',
    'Fiber, total dietary': 'fiber'
}


In [6]:
# =====================================================
# FILTER ONLY REQUIRED NUTRIENTS
# =====================================================

nutrient_df = nutrient_df[
    nutrient_df['nutrient_name'].isin(
        important_nutrients.keys()
    )
]


In [7]:
# =====================================================
# MERGE FOOD_NUTRIENT + NUTRIENT
# =====================================================

merged_df = pd.merge(
    food_nutrient_df,
    nutrient_df,
    on='nutrient_id',
    how='inner'
)


In [8]:
# =====================================================
# PIVOT TABLE
# =====================================================

pivot_df = merged_df.pivot_table(
    index='fdc_id',
    columns='nutrient_name',
    values='amount',
    aggfunc='mean'
).reset_index()

# =====================================================
# RENAME NUTRIENT COLUMNS
# =====================================================

pivot_df.rename(
    columns=important_nutrients,
    inplace=True
)

# =====================================================
# MERGE WITH FOOD TABLE
# =====================================================

final_nutrition_df = pd.merge(
    food_df,
    pivot_df,
    on='fdc_id',
    how='inner'
)


In [9]:

# =====================================================
# HANDLE MISSING VALUES
# =====================================================

nutrition_columns = [
    'calories',
    'protein',
    'carbs',
    'fat',
    'fiber'
]

final_nutrition_df[nutrition_columns] = (
    final_nutrition_df[nutrition_columns]
    .fillna(0)
)

In [10]:

# =====================================================
# REMOVE DUPLICATES
# =====================================================

final_nutrition_df.drop_duplicates(
    subset=['food'],
    inplace=True
)


In [11]:

# =====================================================
# TAKE FIRST 3000 ROWS
# =====================================================

final_nutrition_df = final_nutrition_df.head(3000)

# =====================================================
# DISPLAY OUTPUT
# =====================================================

print("\nFINAL DATASET:")
print(final_nutrition_df.head())

print("\nDATASET SHAPE:")
print(final_nutrition_df.shape)

print("\nCOLUMN NAMES:")
print(final_nutrition_df.columns)



FINAL DATASET:
    fdc_id                food  carbs  calories  fiber  protein   fat
0   319877              Hummus    0.0       0.0    0.0      0.0  19.0
5   319915  Hummus - NFY12140Q    0.0       0.0    5.7      0.0   0.0
7   319934  Hummus - NFY12141F    0.0       0.0    5.4      0.0   0.0
9   319951  Hummus - NFY1213ZX    0.0       0.0    5.5      0.0   0.0
11  319973  Hummus - NFY12143E    0.0       0.0    5.7      0.0   0.0

DATASET SHAPE:
(2461, 7)

COLUMN NAMES:
Index(['fdc_id', 'food', 'carbs', 'calories', 'fiber', 'protein', 'fat'], dtype='str')


In [12]:

# =====================================================
# SAVE CLEAN DATASET
# =====================================================

output_path = r"/home/cnssec/Downloads/diet/Cleaned_data/cleaned_nutrition_dataset.csv"

final_nutrition_df.to_csv(
    output_path,
    index=False
)

print("\nNUTRITION DATASET CREATED SUCCESSFULLY")
print(f"\nSaved at: {output_path}")


NUTRITION DATASET CREATED SUCCESSFULLY

Saved at: /home/cnssec/Downloads/diet/Cleaned_data/cleaned_nutrition_dataset.csv
